# TP — Bras robot à articulation flexible : synthèse LMI robuste

**Commande non linéaire — Master 3EA, parcours ASI — Université Claude Bernard Lyon 1**

Ce notebook accompagne l'énoncé `TP_BrasFlexible.pdf`. Il **est** votre rapport : répondez aux questions dans les cellules de texte, commentez chaque figure.

> **Sur Colab :** commencez par *Fichier > Enregistrer une copie dans Drive*.

Les cellules marquées `# À COMPLÉTER` sont à écrire. Ne modifiez pas les autres sans le signaler.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.signal import place_poles
from scipy.optimize import brentq, minimize_scalar

try:
    import cvxpy as cp
except ImportError:          # cvxpy est installé sur Colab ; sinon :
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'cvxpy'])
    import cvxpy as cp

np.set_printoptions(precision=3, suppress=True)
plt.rcParams['figure.figsize'] = (10, 3.6)

## 0. Le modèle (fourni)

État $x=(\theta_m,\ \dot\theta_m,\ \theta_l,\ \dot\theta_l)$, angle du bras compté depuis la verticale haute. Forme de Lur'e du cours :

$$\dot x = Ax + Bu - G\,\phi(Cx),\qquad \phi(y) = -c\sin y,\qquad c_0 = 3{,}33 .$$

In [ ]:
A = np.array([[0, 1, 0, 0],
              [-48.6, -1.25, 48.6, 0],
              [0, 0, 0, 1],
              [19.5, 0, -19.5, 0]])
B = np.array([[0], [21.6], [0], [0]])
G = np.array([[0], [0], [0], [1.0]])
C = np.array([[0, 0, 1.0, 0]])
c0 = 3.33

def bras(t, x, K, c, umax=None):
    """Dynamique en boucle fermée avec u = K x (saturé si umax est donné)."""
    u = float((K @ x)[0]) if K is not None else 0.0
    if umax is not None:
        u = np.clip(u, -umax, umax)
    return A @ x + B[:, 0] * u + G[:, 0] * c * np.sin(x[2])

def simule(K, c, theta0_deg, T=15.0, umax=None):
    """Simule depuis le bras immobile, ressort au repos, à l'angle theta0 (en degrés)."""
    th = np.deg2rad(theta0_deg)
    x0 = np.array([th, 0.0, th, 0.0])
    sol = solve_ivp(bras, (0, T), x0, args=(K, c, umax), max_step=0.01, rtol=1e-8, atol=1e-10)
    u = (K @ sol.y)[0] if K is not None else np.zeros_like(sol.t)
    if umax is not None:
        u = np.clip(u, -umax, umax)
    return sol.t, sol.y, u

def trace(resultats, titre):
    """resultats : liste de (étiquette, t, x, u)."""
    fig, ax = plt.subplots(1, 2)
    for lab, t, x, u in resultats:
        ax[0].plot(t, np.rad2deg(x[2]), label=lab)
        ax[1].plot(t, u, label=lab)
    ax[0].set(xlabel='t (s)', ylabel=r'$\theta_l$ (deg)', title=titre)
    ax[1].set(xlabel='t (s)', ylabel='u', title='commande')
    for a in ax:
        a.grid(True); a.legend(fontsize=8)
    plt.tight_layout(); plt.show()

## Animation (fournie)
La fonction `anime` transforme une simulation en petit film. Elle sert à *voir* ce que fait la loi de commande ; elle n'est pas à modifier, mais vous pouvez l'appeler sur n'importe laquelle de vos simulations (utile pour la soutenance).

In [ ]:
from matplotlib import animation
from matplotlib import patches as mpatches
from IPython.display import HTML

def _film(fig, dessine, n_images, ips=20):
    """Fabrique le film : vidéo si ffmpeg est disponible (c'est le cas sur Colab), lecteur intégré sinon."""
    anim = animation.FuncAnimation(fig, dessine, frames=n_images, interval=1000 / ips, blit=False)
    plt.close(fig)
    try:
        return HTML(anim.to_html5_video())
    except Exception:
        return HTML(anim.to_jshtml())

def _instants(t, n_images):
    """Indices de n_images instants régulièrement espacés (le solveur, lui, a un pas variable)."""
    return np.clip(np.searchsorted(t, np.linspace(t[0], t[-1], n_images)), 0, len(t) - 1)

def anime(t, x, u, titre='', u_max=5.0, n_images=100):
    """Film du bras (fourni). t, x, u : sorties de simule. Trait orange : repère du moteur ;
    arc coloré : torsion du ressort ; barre du bas : commande u (rouge au-delà de u_max)."""
    k = _instants(t, n_images)
    fig, ax = plt.subplots(figsize=(5.2, 5.6))
    ax.set(xlim=(-1.7, 1.7), ylim=(-2.0, 1.7), title=titre); ax.set_aspect('equal'); ax.axis('off')
    ax.plot([0, 0], [0, 1.55], ':', color='0.6'); ax.text(0.04, 1.5, 'verticale', color='0.5', fontsize=8)
    ax.add_patch(mpatches.Circle((0, 0), 0.34, fc='0.85', ec='k', zorder=2))
    moteur, = ax.plot([], [], color='darkorange', lw=4, solid_capstyle='round', zorder=3)
    bras, = ax.plot([], [], color='0.25', lw=6, solid_capstyle='round', zorder=4)
    charge = mpatches.Circle((0, 0), 0.13, fc='C0', ec='k', zorder=5); ax.add_patch(charge)
    ressort, = ax.plot([], [], lw=3, zorder=3)
    sillage, = ax.plot([], [], color='C0', lw=0.8, alpha=0.5)
    ax.add_patch(mpatches.Rectangle((-1.5, -1.8), 3.0, 0.16, fc='0.93', ec='0.6'))
    ax.plot([0, 0], [-1.84, -1.6], color='k', lw=1)
    barre = mpatches.Rectangle((0, -1.8), 0.0, 0.16, fc='C2'); ax.add_patch(barre)
    ax.text(-1.5, -1.98, '-%g' % u_max, fontsize=8, ha='center'); ax.text(1.5, -1.98, '+%g' % u_max, fontsize=8, ha='center')
    ax.text(0, -1.98, 'commande u', fontsize=8, ha='center')
    texte = ax.text(0.02, 0.98, '', transform=ax.transAxes, va='top', family='monospace', fontsize=9)
    bulle = ax.text(0.98, 0.98, '', transform=ax.transAxes, va='top', ha='right', fontsize=12, weight='bold', color='C2')
    bout = lambda a, L: (L * np.sin(a), L * np.cos(a))            # angle compté depuis la verticale haute

    def dessine(n):
        i = k[n]
        tm, tl = x[0, i], x[2, i]
        bx, by = bout(tl, 1.25)
        bras.set_data([0, bx], [0, by]); charge.center = (bx, by)
        moteur.set_data([0, bout(tm, 0.34)[0]], [0, bout(tm, 0.34)[1]])
        a = np.linspace(tm, tl, 20)
        ressort.set_data(0.5 * np.sin(a), 0.5 * np.cos(a))
        ressort.set_color(plt.cm.coolwarm(0.5 + np.clip((tm - tl) / 0.6, -0.5, 0.5)))
        j0 = max(0, i - 400); sillage.set_data(1.25 * np.sin(x[2, j0:i + 1]), 1.25 * np.cos(x[2, j0:i + 1]))
        ui = u[i]
        barre.set_width(1.5 * np.clip(ui / u_max, -1, 1)); barre.set_facecolor('C3' if abs(ui) > u_max else 'C2')
        texte.set_text('t = %4.1f s\nbras    %6.1f°\nmoteur  %6.1f°\ntorsion %6.1f°\nu = %6.2f'
                       % (t[i], np.rad2deg(tl), np.rad2deg(tm), np.rad2deg(tm - tl), ui))
        immobile = np.hypot(x[1, i], x[3, i]) < 0.02
        if abs(tl) < 0.03 and immobile:
            bulle.set_text('debout !'); bulle.set_color('C2')
        elif immobile and t[i] > 1.0 and abs(np.sin(tl)) > 0.1:
            bulle.set_text('coincé à %.0f°' % np.rad2deg(tl)); bulle.set_color('C3')
        else:
            bulle.set_text('')
    return _film(fig, dessine, n_images)

# Séance 1 — Analyse et synthèse linéaire

### Q3. Secteur de la gravité
Calculer numériquement $s_\star = \min_{y} \sin y / y$ et l'angle où il est atteint.

In [ ]:
# À COMPLÉTER : s_star = min sin(y)/y (indice : minimize_scalar sur un intervalle bien choisi)
y_star, s_star = None, None
print('s* =', s_star, ' atteint en y =', y_star, ' tan(y) - y =', None if y_star is None else np.tan(y_star) - y_star)

### Q2 et Q4. Linéarisé, stabilité, commandabilité

In [ ]:
# À COMPLÉTER : linéarisés en position haute et basse, matrice de commandabilité
A_haut, A_bas, Ctrb = None, None, None
print('valeurs propres, bras dressé  :', np.linalg.eigvals(A_haut))
print('valeurs propres, bras pendant :', np.linalg.eigvals(A_bas))
print('rang de commandabilité :', np.linalg.matrix_rank(Ctrb))

### Q5. Boucle ouverte

In [ ]:
t, x, u = simule(None, c0, 5, T=10)
fig, ax = plt.subplots(1, 2)
ax[0].plot(t, np.rad2deg(x[2]), label=r'$\theta_l$ (bras)')
ax[0].plot(t, np.rad2deg(x[0]), '--', label=r'$\theta_m$ (moteur)')
ax[1].plot(t, np.rad2deg(x[2] - x[0]), label=r'$\theta_l-\theta_m$ (torsion)')
for a in ax:
    a.set_xlabel('t (s)'); a.set_ylabel('deg'); a.grid(True); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

*Votre commentaire :*

### Q6. Synthèse linéaire naïve par placement de pôles
`place_poles(A, B, p)` renvoie $F$ tel que $A - BF$ a les pôles $p$ : avec notre convention $u = Kx$, il faut $K = -F$.

In [ ]:
# À COMPLÉTER
K0 = None
print('K0 =', K0)
print('pôles vérifiés :', np.linalg.eigvals(A_haut + B @ K0))
trace([('%d°' % th, *simule(K0, c0, th)) for th in (10, 90, 180)], 'K0, charge nominale c = c0')

*Votre commentaire :*

### Q7. La charge change : $c = 2c_0$

In [ ]:
trace([('%d°' % th, *simule(K0, 2 * c0, th, T=20)) for th in (10, 90, 180)], 'K0, charge c = 2 c0')
t, x, u = simule(K0, 2 * c0, 10, T=40)
print('état final : theta_m = %.1f°, theta_l = %.1f°' % (np.rad2deg(x[0, -1]), np.rad2deg(x[2, -1])))

In [ ]:
# À COMPLÉTER : écrire re_max(K, c) puis trouver le seuil de charge c_seuil (brentq)
def re_max(K, c):
    ...
c_seuil = None
print('seuil : c = %.3f, soit %.2f c0' % (c_seuil, c_seuil / c0))
cs = np.linspace(0, 3 * c0, 200)
plt.plot(cs / c0, [re_max(K0, c) for c in cs]); plt.axhline(0, color='k', lw=0.8)
plt.xlabel('c / c0'); plt.ylabel('max Re(valeurs propres)'); plt.grid(True); plt.title('K0 : stabilité du bras dressé selon la charge'); plt.show()

In [ ]:
# Film : le gain K0 avec la charge doublée, redressement depuis 180°
anime(*simule(K0, 2 * c0, 180, T=12), titre='K0, charge 2 c0')

*Votre commentaire :*

### Q8. Pentes gelées
On remplace $c_0\sin x_3$ par une droite $s\,x_3$ de pente **constante** $s \in [s_\star c_0,\ c_0]$ (« pente gelée ») : la boucle devient linéaire, $\dot x = (A + sGC + BK_0)\,x$. Le code ci-dessous calcule la pire partie réelle des valeurs propres sur toutes ces pentes.

Répondez ensuite aux points (b) et (c) de l'énoncé : ce test prouve-t-il la stabilité de la vraie boucle, où la « pente » $c_0\sin x_3/x_3$ change le long de la trajectoire ? Qu'apporte une fonction de Lyapunov **commune** à toutes les pentes ?

In [ ]:
pentes = np.linspace(s_star * c0, c0, 200)
pire = max(np.max(np.linalg.eigvals(A + s * G @ C + B @ K0).real) for s in pentes)
print('pire partie réelle sur les pentes gelées : %.3f' % pire)

*Votre réponse :*

# Séance 2 — Synthèse LMI robuste

### Q11. La LMI du cours
Pour un secteur $[k_1,k_2]$ : $\tilde A = A - k_1 GC$, $\bar k = k_2 - k_1$, et l'on cherche $Q \succ 0$, $Y$ tels que

$$\begin{pmatrix}\mathrm{He}(\tilde AQ+BY) + 2\alpha Q & -G+\tfrac12\bar k\,QC^\top\\ -G^\top+\tfrac12\bar k\,CQ & -1\end{pmatrix}\prec 0,\qquad K = YQ^{-1}.$$

Le critère (minimiser $t$ avec $\begin{psmallmatrix}Q & Y^\top\\ Y & t\end{psmallmatrix}\succeq 0$) évite des gains inutilement grands. Il reste à écrire la matrice `M`.

In [ ]:
def secteur(c_max):
    """Secteur [k1, k2] couvrant phi(y) = -c sin y pour toute charge c dans [0, c_max]."""
    return -c_max, -s_star * c_max

def synthese_lmi(k1, k2, alpha=0.0):
    """Renvoie (K, P) avec P = Q^{-1}, ou (None, None) si la LMI est infaisable."""
    At = A - k1 * G @ C
    kb = k2 - k1
    Q = cp.Variable((4, 4), symmetric=True)
    Y = cp.Variable((1, 4))
    t = cp.Variable()
    # À COMPLÉTER : la matrice M (5 x 5) de la LMI, avec cp.bmat
    M = None
    contraintes = [Q >> 1e-4 * np.eye(4),
                   M << -1e-7 * np.eye(5),
                   cp.bmat([[Q, Y.T], [Y, t * np.ones((1, 1))]]) >> 0]
    pb = cp.Problem(cp.Minimize(t), contraintes)
    try:
        pb.solve()
    except cp.error.SolverError:
        return None, None
    if pb.status not in ('optimal', 'optimal_inaccurate'):
        return None, None
    P = np.linalg.inv(Q.value)
    return Y.value @ P, P

In [ ]:
K1, P1 = synthese_lmi(*secteur(c0))
K2, P2 = synthese_lmi(*secteur(2 * c0))
print('K (c_max = c0)   =', K1)
print('K (c_max = 2 c0) =', K2)

In [ ]:
# Film : la loi LMI avec la charge doublée, même départ
anime(*simule(K2, 2 * c0, 180, T=12), titre='loi LMI, charge 2 c0')

### Q12. Vérifications pour $c_{\max} = 2c_0$

In [ ]:
k1, k2 = secteur(2 * c0)
pire = max(np.max(np.linalg.eigvals(A - s * G @ C + B @ K2).real) for s in np.linspace(k1, k2, 300))
print('a) pire partie réelle sur le secteur : %.3f' % pire)
for c in (0, c0, 2 * c0):
    trace([('%d°' % th, *simule(K2, c, th, T=30)) for th in (10, 90, 180)], 'loi LMI, charge c = %.2f c0' % (c / c0))

In [ ]:
# À COMPLÉTER : V(x(t)) = x^T P x le long d'une trajectoire (redressement, c = 2 c0)
t, V = None, None
plt.semilogy(t, V); plt.xlabel('t (s)'); plt.ylabel('V(x(t))'); plt.grid(True); plt.title('Fonction de Lyapunov le long du redressement'); plt.show()
print('V est décroissante :', bool(np.all(np.diff(V) <= 1e-12 * V[:-1])))

*Votre commentaire :*

### Q13. Jusqu'où peut-on charger le bras ?
**(a) Sans limite d'effort.** Dichotomie sur $c_{\max}$ avec la LMI de la question 11.

In [ ]:
# À COMPLÉTER : dichotomie sur c_max (en multiples de c0) ; lo = plus grande valeur faisable trouvée
lo, hi = None, None
print('LMI faisable au moins jusqu\'à c_max = %.1f c0 (borne de recherche : %.0f c0)' % (lo, hi))

**(b) Avec un moteur limité.** On veut que le redressement depuis $\theta_l(0)=180^\circ$ soit garanti avec $|u(t)|\le u_{\max}$. On ajoute deux contraintes à la LMI :

- l'état initial $x_0$ est dans l'ellipsoïde $\mathcal E=\{x^\top Q^{-1}x\le1\}$ : $\begin{pmatrix}1 & x_0^\top\\ x_0 & Q\end{pmatrix}\succeq0$ ;
- sur $\mathcal E$, $|Kx|^2\le t$ : $\begin{pmatrix}Q & Y^\top\\ Y & t\end{pmatrix}\succeq0$.

Comme $V$ décroît, la trajectoire reste dans $\mathcal E$ : $\sqrt t$ est un effort **garanti**. La taille de $Q$ étant maintenant fixée par $x_0$, on ne peut plus normaliser le multiplicateur de la S-procédure à 1 : il devient une variable $\sigma>0$ (déjà en place dans le code).

In [ ]:
def synthese_lmi_effort(k1, k2, theta0_deg=180.0, alpha=0.0):
    """LMI de secteur, avec x0 (bras immobile à theta0) dans l'ellipsoïde {x^T P x <= 1} ;
    minimise l'effort garanti sur cet ellipsoïde. Renvoie (K, P, u_gar), ou (None, None, inf)."""
    At = A - k1 * G @ C
    kb = k2 - k1
    th = np.deg2rad(theta0_deg)
    x0 = np.array([[th], [0.0], [th], [0.0]])
    Q = cp.Variable((4, 4), symmetric=True)
    Y = cp.Variable((1, 4))
    t = cp.Variable((1, 1))
    sigma = cp.Variable((1, 1))                      # multiplicateur de la S-procédure
    H = At @ Q + B @ Y
    hors_diag = -G @ sigma + 0.5 * kb * Q @ C.T
    M = cp.bmat([[H + H.T + 2 * alpha * Q, hors_diag],
                 [hors_diag.T, -sigma]])
    contraintes = [Q >> 1e-4 * np.eye(4), sigma >= 1e-6, M << -1e-7 * np.eye(5)]
    # À COMPLÉTER : ajouter à `contraintes` les deux LMI ci-dessus (cp.bmat ; x0 est une colonne 4 x 1)
    pb = cp.Problem(cp.Minimize(t[0, 0]), contraintes)
    try:
        pb.solve()
    except cp.error.SolverError:
        return None, None, np.inf
    if pb.status not in ('optimal', 'optimal_inaccurate'):
        return None, None, np.inf
    P = np.linalg.inv(Q.value)
    return Y.value @ P, P, float(np.sqrt(t.value[0, 0]))

print(' c_max    effort garanti   max|u| simulé (redressement, c = c_max)')
for mult in (1, 2, 3, 4, 5):
    K, P, u_gar = synthese_lmi_effort(*secteur(mult * c0))
    if K is None:
        print('%3d c0   infaisable' % mult); continue
    t_, x_, u_ = simule(K, mult * c0, 180, T=30)
    print('%3d c0      %6.2f          %6.2f' % (mult, u_gar, np.abs(u_).max()))


In [ ]:
# À COMPLÉTER : plus grande charge c_lim (en multiples de c0) pour laquelle l'effort garanti reste <= 5 (brentq)
u_max = 5.0
c_lim = None
print('avec |u| <= %.0f garanti, redressement certifié jusqu\'à c_max =' % u_max, c_lim, 'c0')


*Votre réponse (comparer (a) et (b) ; l'effort garanti est-il proche de l'effort simulé ?) :*

### Q14. Le prix de la rapidité

In [ ]:
lignes = []
for alpha in (0, 0.5, 1, 2, 3, 4, 5):
    K, P = synthese_lmi(*secteur(2 * c0), alpha=alpha)
    if K is None:
        lignes.append((alpha, np.nan, np.nan, np.nan)); continue
    t, x, u = simule(K, c0, 180)
    au_dessus = np.where(np.abs(x[2]) > 0.05 * np.pi)[0]
    t5 = t[au_dessus[-1]] if len(au_dessus) else 0.0
    lignes.append((alpha, np.linalg.norm(K), np.abs(u).max(), t5))
print(' alpha   |K|     max|u|   t(5%)')
for l in lignes:
    print('%5.1f  %6.1f  %7.1f  %6.2f s' % l)

*Votre réponse (choix de $\alpha$ pour $|u|\le 5$) :*

### Question ouverte (une au choix, obligatoire)
Choisissez une des trois questions de l'énoncé (charge qui varie dans le temps, secteur local, comparaison avec la linéarisation par bouclage) et traitez-la ici. Elle n'est pas guidée et doit apparaître dans la soutenance.

In [ ]:
# Votre travail